# Compare analytical method with simulation (SI, fig. 14):

Here we show that our analytical method to compute the STDDs (see section 4.2.1 and the `stddc`-package) is consistent with the results of numerical simulations (computed by the `neuralsampling`-package based on eq. 8ff)

In [ ]:
import os

os.environ["NUMBA_DISABLE_JIT"] = "1"

In [ ]:
from pathlib import Path
from functools import partial

import numpy as np
from numba import njit, vectorize
import matplotlib as mpl
import matplotlib.pyplot as plt

from stddc import STDDMaker, alpha_PSP, rect_PSP, exp_window
from neuralsampling import network
from neuralsampling import stdp_functions

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# define the style etc.
mpl.style.use("../../mystyle.mpl")

In [ ]:
FIG_DIR = Path("../../figs/sal_principle")
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
T_REF = 30
T_SYN_ALPHA = 10
T_MAX_ANA = 60
T_MAX_SIM = 1_000_000


_alpha_kernel = vectorize(alpha_PSP)


@njit
def alpha_kernel(t, tau_syn):
    return _alpha_kernel(t, T_REF, tau_syn)


def worker(psps, b, w, tau_syn):
    sm = STDDMaker(
        psps[0],
        t_max=T_MAX_ANA,
        t_ref=T_REF,
        t_syn=tau_syn,
        w_12=w[0, 1],
        w_21=w[1, 0],
        b_1=b[0],
        b_2=b[1],
    )
    stdd = sm.calc_stdd()
    spks = network.sim_poisson_neurons(
        t_max=T_MAX_SIM,
        psp_kernel=psps[1],
        bias=b,
        weights=w,
        t_ref=T_REF,
        tau_syn=tau_syn,
    )
    stds, _ = stdp_functions.get_first_order_stds_2nrn(spks, 2)
    stdd_sim, bins = np.histogram(stds, bins=121, range=(-60.5, 60.5), density=True)
    return stdd, sm.times, stdd_sim, bins

In [ ]:
b = np.array([-0.4, 0.3])
w = np.array([[0.0, 1.2], [0.8, 0.0]])
# stdd, dts, stdd_sim, dts_sim = worker((alpha_PSP, alpha_kernel), b, w, 10)

spks = network.sim_poisson_neurons(
    t_max=T_MAX_SIM, psp_kernel=alpha_kernel, bias=b, weights=w, t_ref=T_REF, tau_syn=10
)

In [ ]:
fig, ax = plt.subplots(2, 2, sharex="row", sharey=True, figsize=(8 / 2.54, 8 / 2.54))

rng = np.random.default_rng()

b = rng.uniform(low=-1.0, high=1.0, size=2)
w = rng.uniform(low=-1.0, high=1.0, size=(2, 2))
np.fill_diagonal(w, 0)
stdd, dts, stdd_sim, dts_sim = worker((rect_PSP, network.rect_kernel), b, w, T_REF)
ax[0, 0].stairs(stdd_sim, dts_sim, fill=True, label="simulation")
ax[0, 0].plot(dts, stdd, label="analytical")

b = rng.uniform(low=-1.0, high=1.0, size=2)
w = rng.uniform(low=-1.0, high=1.0, size=(2, 2))
np.fill_diagonal(w, 0)
stdd, dts, stdd_sim, dts_sim = worker((rect_PSP, network.rect_kernel), b, w, T_REF)
ax[1, 0].stairs(stdd_sim, dts_sim, fill=True)
ax[1, 0].plot(dts, stdd)

b = rng.uniform(low=-1.0, high=1.0, size=2)
w = rng.uniform(low=-1.0, high=1.0, size=(2, 2))
np.fill_diagonal(w, 0)
stdd, dts, stdd_sim, dts_sim = worker((alpha_PSP, alpha_kernel), b, w, T_SYN_ALPHA)
ax[0, 1].stairs(stdd_sim, dts_sim, fill=True)
ax[0, 1].plot(dts, stdd)

b = rng.uniform(low=-1.0, high=1.0, size=2)
w = rng.uniform(low=-1.0, high=1.0, size=(2, 2))
np.fill_diagonal(w, 0)
stdd, dts, stdd_sim, dts_sim = worker((alpha_PSP, alpha_kernel), b, w, T_SYN_ALPHA)
ax[1, 1].stairs(stdd_sim, dts_sim, fill=True)
ax[1, 1].plot(dts, stdd)

# Iterate over all subplots to customize the appearance
for i in range(2):
    for j in range(2):
        a = ax[i, j]

        # Remove top, right, and left spines
        a.spines["top"].set_visible(False)
        a.spines["right"].set_visible(False)
        a.spines["left"].set_visible(False)

        # Remove y-axis ticks and labels
        a.set_yticks([])

for i in range(2):
    ax[1, i].set_xlabel(r"$\Delta t$")
    ax[1, i].set_xticks(
        [-T_REF, 0, T_REF],
        labels=[r"$-\tau_\mathrm{ref}$", "0", r"$\tau_\mathrm{ref}$"],
    )
    ax[i, 0].set_ylabel(r"$p(\Delta t)$")

ax[0, 0].set_title("rect PSP")
ax[0, 1].set_title("alpha PSP")

# Get handles and labels from the first subplot
handles, labels = ax[0, 0].get_legend_handles_labels()

# Create a figure-level legend at the top center
fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.0))

# Adjust layout to make room for the legend
plt.tight_layout()
plt.subplots_adjust(top=0.80)

In [ ]:
fig.savefig(FIG_DIR / "stdd_ana_vs_sim.pdf")
fig.savefig(FIG_DIR / "stdd_ana_vs_sim.png")
fig.savefig(FIG_DIR / "stdd_ana_vs_sim.svg")